[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/ml-curriculum/02_linear_regression/02_linear_regression_solutions.ipynb)

# 02. Linear Regression — 연습 문제 해설

[02_linear_regression.ipynb](02_linear_regression.ipynb) 끝의 연습 문제 2개에 대한 정답 코드와 해설입니다. **먼저 직접 시도해본 뒤** 참고하세요.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q scikit-learn numpy matplotlib

import numpy as np
import matplotlib.pyplot as plt

## 연습 1. 학습률(lr)을 0.1, 0.001로 바꿔보기

원본 노트북과 같은 데이터/초기화 조건에서 `lr`만 바꿔가며 비교합니다.

In [ ]:
rng = np.random.default_rng(42)
x = np.linspace(0, 10, 50)
y = 2 * x + 3 + rng.normal(0, 1.5, size=x.shape)

def gradient_descent(x, y, lr, epochs=200):
    W, b = 0.0, 0.0
    m = len(x)
    history = []
    for _ in range(epochs):
        y_pred = W * x + b
        cost = np.mean((y_pred - y) ** 2)
        history.append(cost)
        dW = (2 / m) * np.sum((y_pred - y) * x)
        db = (2 / m) * np.sum(y_pred - y)
        W -= lr * dW
        b -= lr * db
        if not np.isfinite(cost) or cost > 1e8:
            break  # 발산 시 조기 종료
    return W, b, history

results = {}
for lr in [0.001, 0.01, 0.1]:
    W, b, hist = gradient_descent(x, y, lr)
    results[lr] = hist
    print(f"lr={lr:<6} 최종 cost={hist[-1]:.4f}  (epochs 실행={len(hist)})  W={W:.3f} b={b:.3f}")

In [ ]:
plt.figure(figsize=(7, 4))
for lr, hist in results.items():
    plt.plot(hist, label=f"lr={lr}")
plt.yscale("log")
plt.xlabel("epoch")
plt.ylabel("cost (log scale)")
plt.title("학습률에 따른 수렴 속도 비교")
plt.legend()
plt.show()

**해설**
- `lr=0.001`: 200 epoch 안에 거의 수렴하지 못합니다 — cost가 아주 천천히 줄어듭니다. 더 많은 epoch이 필요합니다.
- `lr=0.01`(원본 기본값): 200 epoch 안에 안정적으로 잘 수렴합니다.
- `lr=0.1`: 이 데이터/스케일에서는 여전히 수렴하지만 훨씬 빠르게 cost가 줄어듭니다. 다만 `lr`을 계속 키우면(예: 1.0 이상) 그래디언트가 매 스텝마다 목표를 지나쳐버려서(overshoot) cost가 오히려 발산(증가)할 수 있습니다 — 위 코드의 조기 종료 조건이 그 상황을 잡아줍니다.
- 결론: 학습률은 "수렴 속도"와 "안정성" 사이의 트레이드오프입니다. 너무 작으면 느리고, 너무 크면 발산합니다.

## 연습 2. 특성 추가(출석률)로 성능 개선 확인

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

rng = np.random.default_rng(0)
n_samples = 25
quiz1 = rng.integers(60, 100, n_samples)
quiz2 = rng.integers(60, 100, n_samples)
midterm = rng.integers(60, 100, n_samples)
attendance = rng.integers(70, 101, n_samples)  # 새 특성: 출석률(%)

# 기말 점수에 출석률도 실제로 영향을 주도록 시뮬레이션
final = (0.25 * quiz1 + 0.25 * quiz2 + 0.3 * midterm + 0.2 * attendance
         + rng.normal(0, 3, n_samples)).round(1)

X_base = np.column_stack([quiz1, quiz2, midterm])
X_ext = np.column_stack([quiz1, quiz2, midterm, attendance])
y = final

def fit_and_eval(X, y, label):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
    reg = LinearRegression().fit(X_train, y_train)
    pred = reg.predict(X_test)
    print(f"{label:>12}: MSE={mean_squared_error(y_test, pred):.3f}  R^2={r2_score(y_test, pred):.3f}")

fit_and_eval(X_base, y, "3개 특성")
fit_and_eval(X_ext, y, "4개 특성(+출석률)")

**해설**
- 출석률이 실제로 기말 점수와 관련이 있도록 데이터를 만들었기 때문에, 이 특성을 추가하면 보통 **MSE는 낮아지고 R²는 높아집니다** (모델이 설명하는 분산 비율이 늘어남).
- 주의: 실제 데이터에서는 관련 없는 특성을 추가하면 오히려 노이즈가 늘어나 성능이 나빠지거나 과적합될 수 있습니다. "특성을 늘리면 무조건 좋다"가 아니라, **타깃과 실제로 관련 있는 특성**을 추가해야 도움이 됩니다.
- 샘플이 25개로 매우 적기 때문에 `random_state`를 바꿔가며 여러 번 돌려보면 결과가 흔들릴 수 있습니다 — 데이터가 적을 때 평가 지표의 분산이 크다는 것도 함께 체감할 수 있습니다.